# Mulitcompartment (stick) Hodgkin Huxley model

In this exercise, you’ll simulate a multicompartment Hodgkin Huxley stick-neuron.<br>
You’ll learn how action potentials propagate in space and how extracellular electric fields can stimulate neurons as in neuroprosthetics.<br>
The governing equation for the nth compartment the multicompartment Hodgkin Huxley model is:<br>

dV_n/dt = (-IIon_n+IStim_n+IAxial_n)/C_n

with 

- IAxial_n = (v_n-1-v_n) / (R_n-1/2+R_n/2) + (v_n+1-v_n) / (R_n+1/2+R_n/2) i.e., multiple single compartment Hodgkin Huxley models are connected by resistors.
 
 Two stimulation modes: <br>
 (1) Current injection into center compartment ('iClamp') <br>
 (2) Extracellular stimulation with a point source electrode ('extracellular')
                  
 ```
                       _
                  (2) / \     ||   
                      \_/    _||_ (1)
                             \  /                    
   -------------------------- \/ --------------------------
   |    1     |    2     |    3     |    4     |    5     |
   --------------------------------------------------------
```

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Stimulus parameters
mode = 'extracellular' # Stimulus mode, 'iClamp' or 'extracellular'
rhoE = 300 # Extracellular resistivity, in Ohm*cm, for 'extracellular' only
cellX = 0 # X-shift of cell relative to the electrode (at 0/0)), in um, for 'extracellular' only
cellY = 25 # Y-shift of cell relative to the electrode (at 0/0)), in um, for 'extracellular' only
I = -4 # Stimulus amplitude, in pA ('iClamp') or uA ('extracellular')

# Temporal parameters
tStop = 10 # Total duration of simulation, in ms
tDel = 1 # Delay until stimulus starts, in ms
tDur = 0.5 # Duration of stimulus, in ms
tDt = 0.0125 # Time step, in ms

# Stick/fiber parameters
lStick = 2000 # Total length of fiber, in um
nComp = 201 # Number of compartments
rComp = 1 # Compartment radius, in um

# Biophysics
c = 1 # Membrane specific capacitance, in uF/cm2, original=1
rhoA = 100 # Axial/Intracellular resistivity, in Ohm*cm, original=100

# Temperature
temp = 6.3 # Model temperature, in Celsius, original=6.3

# Hodgkin & Huxley parameters
vInit = -65 # Membrane voltage to initialize the simulation, in mV, original=-65
gNa = 120 # Sodium channel maximum conductivity, in mS/cm2, original=120
gK = 36 # Potassium channel maximum conductivity, in mS/cm2, original=36
gL = 0.3 # Leak channel maximum conductivity, in mS/cm2, original=0.3
eNa = 50 # Sodium reversal/equilibrium potential, in mV, original=50
eK = -77 # Potassium reversal/equilibrium potential, in mV, original=-77
eL = -54.3 # Leak reversal/equilibrium potential, in mV, original=-54.3

## Step 1: Euler Integration

We’ll use a backward (implicit) Euler’s method to numerically integrate the differential equation:

In [ ]:
### ------------- DEFINE FUNCTIONS FOR ALPHAS AND BETAS -------------
def alphaM(v,kTemp):
    return kTemp*(0.1*(v+40))/(1-np.exp(-(v+40)/10))
def betaM(v,kTemp):
    return kTemp*4*np.exp(-(v+65)/18)
def alphaH(v,kTemp):
    return kTemp*0.07*np.exp(-(v+65)/20)
def betaH(v,kTemp):
    return kTemp*1/(1+np.exp(-(v+35)/10))
def alphaN(v,kTemp):
    return kTemp*(0.01*(v+55))/(1-np.exp(-(v+55)/10))
def betaN(v,kTemp):
    return kTemp*0.125*np.exp(-(v+65)/80)


### ------------- COMPUTE ADDITIONAL PARAMETERS -------------
# Compartment length based on total length and number of compartments, in um
lComp = lStick/(nComp-1) 
# Central compartment of stick
centerComp = int((nComp-1)/2+1)

# Axial resistance of each compartment, in kOhm
R = ((2*rhoA*lComp*1e-4)/(2*(rComp*1e-4)**2*np.pi))*1e-3
# Compartment surface, in cm2
A = (2*rComp*np.pi*lComp)*1e-8
# Compartment membrane capacitance, in uF
C = c*A

# Time step
timeSteps = int(tStop/tDt)+1
timeStep = np.linspace(0,tStop,timeSteps)

# Temperature adjustment
kTemp = 3**((temp-6.3)/10)

# Other constants
vAdd = 0.001

# Set up tridiagonal matrix for fast computation of iAxial
matInvDiag = np.concatenate(([1+(tDt/C)*(1/R)],np.ones(nComp-2)+(tDt/C)*(2/R),[1+(tDt/C)*(1/R)])) # in (ms/uF)*(1/kOhm)==1/V
matInvOffDiag = np.ones(nComp-1)*-(tDt/C)*(1/R)
matInv = np.diag(matInvDiag,0) + np.diag(matInvOffDiag,-1) + np.diag(matInvOffDiag,1)

# Set up tridiagonal matrix for iStim (extracellular) computation
matAxDiag = np.concatenate(([-1/R],np.ones(nComp-2)*-2/R,[-1/R])) # in 1/kOhm
matAxOffDiag = np.ones(nComp-1)*1/R
matAxial = np.diag(matAxDiag,0) + np.diag(matAxOffDiag,-1) + np.diag(matAxOffDiag,1)

# Stimulus currents depending on stimulus type
if mode=='iClamp': # Simple current conversion
    iStim = np.zeros(nComp)
    iStim[centerComp-1] = 1e-6*I/A # in 1e-6*pA/cm2==uA/cm2
elif mode=='extracellular':
    # Compute potentials at compartment centers, electrode is located at 0/0
    x = np.linspace(-(centerComp-1)*lComp,(centerComp-1)*lComp,nComp)+cellX
    y = np.ones(nComp)*cellY
    # Compute extracellular potentials for point source electrode
    # Euklidean distance for each compartment center
    compDist = 1e-4*np.sqrt(x**2+y**2) # in cm
    # Analytical potentials (point source) for given distance
    potentials = 1e-3*(rhoE*I)/(4*np.pi*compDist) # in 1e-3*(Ohm*cm*uA)/cm==mV
    
    # Matrix x vector for stimulus current
    iStim = np.matmul(matAxial,potentials)/A # in ((1/kOhm)*mV)/cm2==uA/cm2

# Compute initial values
v0 = vInit
m0 = alphaM(v0,kTemp)/(alphaM(v0,kTemp)+betaM(v0,kTemp))
h0 = alphaH(v0,kTemp)/(alphaH(v0,kTemp)+betaH(v0,kTemp))
n0 = alphaN(v0,kTemp)/(alphaN(v0,kTemp)+betaN(v0,kTemp))

# Allocate memory for v, m, h and n
vMat = np.zeros((nComp,timeSteps))
mMat = np.zeros((nComp,timeSteps))
hMat = np.zeros((nComp,timeSteps))
nMat = np.zeros((nComp,timeSteps))

# Set initial values
vMat[:,0] = v0
mMat[:,0] = m0
hMat[:,0] = h0
nMat[:,0] = n0


### --------- SOLVE ODE & POSTPROCESSING ---------
for t in range(0,timeSteps-1):
    
    # States at current time step
    vVecT = vMat[:,t]
    mVecT = mMat[:,t]
    hVecT = hMat[:,t]
    nVecT = nMat[:,t]

    # Stimulus current
    iStimVec = np.zeros(nComp)
    if t>=int(tDel/tDt) and t<int((tDel+tDur)/tDt):
        iStimVec = iStim # in uA/cm2
        
    # Ionic currents
    # Sodium
    iNaVec = gNa*mVecT**3*hVecT*(vVecT-eNa) # in (mS/cm2)*mV==uA/cm2
    # Potassium
    iKVec = gK*nVecT**4*(vVecT-eK) # in (mS/cm2)*mV==uA/cm2
    # Leak
    iLVec = gL*(vVecT-eL) # in (mS/cm2)*mV==uA/cm2
    # Sum
    iIonVec = iNaVec+iKVec+iLVec # in uA/cm2    
    
    # Update gating variables with new v
    # Additonal ionic contribution needed for BE
    # Sodium
    iNaAuxVec = gNa*mVecT**3*hVecT*(vVecT+vAdd-eNa) # in (mS/cm2)*mV==uA/cm2
    # Potassium
    iKAuxVec = gK*nVecT**4*(vVecT+vAdd-eK) # in (mS/cm2)*mV==uA/cm2
    # Leak
    iLAuxVec = gL*(vVecT+vAdd-eL) # in (mS/cm2)*mV==uA/cm2
    # Sum
    rhsdidvVec = (iNaAuxVec-iNaVec+iKAuxVec-iKVec+iLAuxVec-iLVec)/vAdd # in (uA/cm2)/mV

    # Compute change of v
    # Right hand side of matrix equation to be solved
    RHS = vVecT+(-iIonVec+rhsdidvVec*vVecT+iStimVec)*(tDt/c)
    # Add ionic current contribution to left hand side
    np.fill_diagonal(matInv,matInvDiag+rhsdidvVec*(tDt/c))
    
    # Solve matrix equation
    vMat[:,t+1] = np.linalg.solve(matInv,RHS) # in mV

    # Restore inverse Matrix (not a very elegant solution...)
    np.fill_diagonal(matInv,matInvDiag-rhsdidvVec*(tDt/c))
    
    # Update gating variables with new v
    mMat[:,t+1] = (mVecT+tDt*alphaM(vMat[:,t+1],kTemp))/(1+tDt*(alphaM(vMat[:,t+1],kTemp)+betaM(vMat[:,t+1],kTemp)))
    hMat[:,t+1] = (hVecT+tDt*alphaH(vMat[:,t+1],kTemp))/(1+tDt*(alphaH(vMat[:,t+1],kTemp)+betaH(vMat[:,t+1],kTemp)))
    nMat[:,t+1] = (nVecT+tDt*alphaN(vMat[:,t+1],kTemp))/(1+tDt*(alphaN(vMat[:,t+1],kTemp)+betaN(vMat[:,t+1],kTemp)))
        

### --------- VISUALIZE RESULTS ---------
# Membrane voltage over time and space
fig = plt.figure()
ax1 = fig.add_subplot(111)
ax1.grid()
ax1.spines["right"].set_visible(False)
ax1.spines["top"].set_visible(False)
ax1.plot(timeStep,vMat.T,'b',lw=0.25)
ax1.plot(timeStep,vMat[centerComp-1,:],'r')
ca = plt.gca()
ca.add_patch(plt.Rectangle((tDel,-100),tDur,10,facecolor='r'))
plt.xlim(0,tStop)
plt.ylim(-100, 60)
plt.xlabel('Time (ms)')
plt.ylabel('Membrane voltage (mV)')
plt.show()

# Membrane voltage over time (spatial)
fig = plt.figure()
ax2 = fig.add_subplot(111)
ax2.spines["right"].set_visible(False)
ax2.spines["top"].set_visible(False)
offset = -vInit;
for i in range(nComp):
    if i==centerComp-1:
        ax2.plot(timeStep,vMat[i,:]+offset,'r');
    else:
        ax2.plot(timeStep,vMat[i,:]+offset,'b',lw=0.25);
    offset = offset-lComp;
ca = plt.gca()
ca.add_patch(plt.Rectangle((tDel,offset-150),tDur,50,facecolor='r'))
plt.xlim(0,tStop)
plt.ylim([offset-150,150])
plt.xlabel('Time (ms)')
plt.ylabel('Location along stick (um)')
plt.show()
    


## Step 2: Tasks

1. Find the spiking threshold for intracellular stimulation (mode = 'iClamp', tolerance/minimum step size == 10 pA). Where does the action potential initiate and how does it propagate along the fiber?
2. Change the intracellular resitivity ('rhoA') to values of 50, 200 and 400 Ohm*cm and describe how action potential propagation changes. Use twice the stimulus amplitude you determined in Task 1.

- Change rhoA back to 100 Ohm*cm for all further simulations.

3. Apply extracellular stimulation (mode = 'extracellular') with an amplitude of -4 uA, i.e., a suprathreshold cathodic pulse. Move the stimulation electrode along the fiber/x-axis (change 'cellX' to values from -750 to 750 um). How does the site of inititation of action potentials change with electrode position?

- Change cellX back to 0 um for all further simulations.

4. Move the stimulation electrode closer and further away from the fiber ('cellY'). Describe qualitatively how threshold changes with distance from the fiber.
5. (Bonus) Compute the activating function (see slides) for anodic and cathodic stimuli and describe the difference. Compare thresholds for anodic and cathodic stimulation, describe, based on the activating function plotted before, why cathodic stimuli result in lower thresholds.